# RAG + SQL Router: Intelligent Hybrid Query Engine

This notebook demonstrates the complete architecture of an intelligent query routing system that automatically classifies user queries and routes them to either:
- **SQL Engine** — for structured city/population data queries
- **RAG Pipeline** — for document-based semantic search with trust scoring

**Tech Stack:** OpenAI GPT-4o-mini | ChromaDB | FastAPI | Cleanlab Codex

## 1. Setup & Configuration

In [ ]:
import os
from openai import OpenAI
from sqlalchemy import create_engine, text, inspect
import chromadb
from chromadb.utils import embedding_functions

# Configuration
os.environ["OPENAI_API_KEY"] = "sk-your-openai-api-key-here"
os.environ["CODEX_API_KEY"] = "your-codex-api-key-here"  # Optional

OPENAI_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
DATABASE_PATH = "backend/data/city_database.sqlite"

client = OpenAI()
print("OpenAI client initialized successfully")

## 2. SQL Database Setup

We connect to a SQLite database containing US city statistics (city_name, population, state).

In [ ]:
# Connect to the SQLite database
engine = create_engine(f"sqlite:///{DATABASE_PATH}")

# Inspect the schema
inspector = inspect(engine)
tables = inspector.get_table_names()
print(f"Tables: {tables}")

for table in tables:
    columns = inspector.get_columns(table)
    print(f"\nTable '{table}' columns:")
    for col in columns:
        print(f"  - {col['name']} ({col['type']})")

In [ ]:
# Preview the data
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM city_stats ORDER BY population DESC LIMIT 10"))
    rows = result.fetchall()
    print("Top 10 cities by population:")
    print(f"{'City':<20} {'State':<15} {'Population':>12}")
    print("-" * 50)
    for row in rows:
        print(f"{row[0]:<20} {row[2]:<15} {row[1]:>12,}")

## 3. Intelligent Query Router

The router uses GPT-4o-mini to classify queries into `sql` or `rag` categories. This is the core intelligence of the system — it examines the user's intent and decides which engine is best suited to answer.

In [ ]:
ROUTER_SYSTEM_PROMPT = """You are an intelligent query router. Your job is to classify user queries into one of two categories:

1. "sql" - Use this when the query is about US city statistics, population data, or state information.
   Examples: "What is the population of Houston?", "Which cities are in California?", "What's the largest city?"

2. "rag" - Use this when the query is about anything else, especially when it requires searching through uploaded documents.
   Examples: "What does the report say about Q4 revenue?", "Summarize the key findings", "What is the weather policy?"

Respond with ONLY the word "sql" or "rag". Nothing else."""


def classify_query(query: str) -> str:
    """Classify a query as 'sql' or 'rag' using GPT-4o-mini."""
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": ROUTER_SYSTEM_PROMPT},
            {"role": "user", "content": query}
        ],
        temperature=0,
        max_tokens=10,
    )
    classification = response.choices[0].message.content.strip().lower()
    return "sql" if "sql" in classification else "rag"


# Test the router
test_queries = [
    "What is the population of Houston, Texas?",
    "Which state has the most cities?",
    "What does the document say about climate change?",
    "List all cities in California",
    "Summarize the key findings from the uploaded report",
]

print("Query Classification Results:")
print("=" * 60)
for q in test_queries:
    route = classify_query(q)
    icon = "🗄️" if route == "sql" else "📄"
    print(f"{icon} [{route.upper()}] {q}")

## 4. Text-to-SQL Engine

For SQL-routed queries, we use GPT-4o-mini to generate SQL from natural language, execute it against the database, and synthesize a human-readable response.

In [ ]:
def get_schema_info() -> str:
    """Get database schema as a string for the LLM."""
    inspector = inspect(engine)
    tables = inspector.get_table_names()
    schema_parts = []
    for table in tables:
        columns = inspector.get_columns(table)
        col_defs = ", ".join([f"{c['name']} ({c['type']})" for c in columns])
        schema_parts.append(f"Table: {table} | Columns: {col_defs}")
    return "\n".join(schema_parts)


SQL_SYSTEM_PROMPT = """You are an expert SQL query generator. Given a natural language question about a SQLite database, generate the appropriate SQL query.

Database Schema:
{schema}

Rules:
1. Generate ONLY the SQL query, no explanations
2. Use SQLite syntax
3. Always use exact column names from the schema
4. For text comparisons, use LIKE with % for partial matches
5. Return useful, readable results
6. Never use DELETE, DROP, UPDATE, INSERT, or any data-modifying statements
7. Only use SELECT statements"""


def text_to_sql(query: str) -> dict:
    """Convert natural language to SQL, execute, and synthesize response."""
    schema = get_schema_info()
    
    # Step 1: Generate SQL
    sql_response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SQL_SYSTEM_PROMPT.format(schema=schema)},
            {"role": "user", "content": query}
        ],
        temperature=0,
        max_tokens=500,
    )
    sql = sql_response.choices[0].message.content.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    
    # Step 2: Execute SQL
    with engine.connect() as conn:
        result = conn.execute(text(sql))
        rows = result.fetchall()
        columns = list(result.keys())
    
    # Step 3: Synthesize natural language response
    data_str = "\n".join([str(dict(zip(columns, row))) for row in rows[:20]])
    
    synthesis = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful data analyst. Given the user's question and SQL results, provide a clear, concise natural language answer."},
            {"role": "user", "content": f"Question: {query}\n\nSQL: {sql}\n\nResults:\n{data_str}"}
        ],
        temperature=0.3,
        max_tokens=500,
    )
    
    return {
        "response": synthesis.choices[0].message.content.strip(),
        "sql_query": sql,
        "data": [dict(zip(columns, row)) for row in rows]
    }

In [ ]:
# Test the SQL engine
result = text_to_sql("What is the population of Houston, Texas?")
print(f"Query: What is the population of Houston, Texas?")
print(f"Generated SQL: {result['sql_query']}")
print(f"Response: {result['response']}")
print(f"Raw Data: {result['data']}")

In [ ]:
# Test with a more complex query
result = text_to_sql("Which are the top 5 most populated cities?")
print(f"Query: Which are the top 5 most populated cities?")
print(f"Generated SQL: {result['sql_query']}")
print(f"Response: {result['response']}")

## 5. RAG Pipeline with ChromaDB

For document queries, we use ChromaDB for vector storage with OpenAI embeddings. Documents are chunked, embedded, and stored for semantic retrieval.

In [ ]:
# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="./chroma_db")

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name=EMBEDDING_MODEL,
)


def chunk_text(text: str, chunk_size: int = 800, overlap: int = 200) -> list:
    """Split text into overlapping chunks for embedding."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


def ingest_document(text: str, source: str, collection_name: str = "demo"):
    """Ingest a document into ChromaDB."""
    collection = chroma_client.get_or_create_collection(
        name=collection_name,
        embedding_function=openai_ef,
        metadata={"hnsw:space": "cosine"}
    )
    
    chunks = chunk_text(text)
    ids = [f"{source}_{i}" for i in range(len(chunks))]
    metadatas = [{"source": source, "chunk_index": i} for i in range(len(chunks))]
    
    collection.add(documents=chunks, ids=ids, metadatas=metadatas)
    return len(chunks)


def query_documents(query: str, collection_name: str = "demo", top_k: int = 3) -> dict:
    """Query documents using semantic search and generate an answer."""
    collection = chroma_client.get_collection(
        name=collection_name,
        embedding_function=openai_ef,
    )
    
    results = collection.query(query_texts=[query], n_results=top_k)
    
    if not results["documents"][0]:
        return {"response": "No relevant information found.", "sources": [], "context": ""}
    
    context_parts = results["documents"][0]
    sources = [m["source"] for m in results["metadatas"][0]]
    context_str = "\n\n---\n\n".join(context_parts)
    
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": (
                "You are a meticulous document analyst. Answer based EXCLUSIVELY on the provided context. "
                "If the context lacks sufficient information, state that clearly."
            )},
            {"role": "user", "content": f"Context:\n{context_str}\n\n---\n\nQuestion: {query}"}
        ],
        temperature=0.2,
        max_tokens=1000,
    )
    
    return {
        "response": response.choices[0].message.content.strip(),
        "sources": list(set(sources)),
        "context": context_str
    }

print("RAG Pipeline ready")

In [ ]:
# Demo: Ingest a sample document
sample_document = """
Climate Change Impact Report 2024

Executive Summary:
Global temperatures have risen by 1.2°C above pre-industrial levels. 
The report finds that extreme weather events have increased by 40% in the last decade.
Sea levels are projected to rise by 0.3-0.6 meters by 2100 under current emission scenarios.

Key Findings:
1. Arctic ice coverage has decreased by 13% per decade since 1979.
2. Ocean acidification has increased by 26% since the industrial revolution.
3. Renewable energy adoption has grown 25% year-over-year globally.
4. Carbon capture technology investment reached $6.4 billion in 2024.

Recommendations:
- Accelerate transition to renewable energy sources
- Invest in carbon capture and storage infrastructure
- Implement stricter emissions regulations for industrial sectors
- Increase funding for climate adaptation in vulnerable regions
"""

num_chunks = ingest_document(sample_document, "climate_report_2024")
print(f"Document ingested: {num_chunks} chunks created")

In [ ]:
# Test RAG query
result = query_documents("What are the key findings about climate change?")
print(f"Query: What are the key findings about climate change?")
print(f"Response: {result['response']}")
print(f"Sources: {result['sources']}")

## 6. Cleanlab Codex Trust Scoring

Cleanlab Codex validates RAG responses by scoring their trustworthiness. This ensures that generated answers are grounded in the source documents and flags potentially unreliable responses.

In [ ]:
def validate_with_codex(query: str, context: str, response: str) -> dict:
    """Validate a RAG response using Cleanlab Codex."""
    try:
        from cleanlab_codex.client import Client
        from cleanlab_codex.project import Project
        
        codex_client = Client()
        project = codex_client.create_project(name="RAG-SQL-Router-Notebook")
        access_key = project.create_access_key("notebook-key")
        codex_project = Project.from_access_key(access_key)
        
        prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"
        messages = [{"role": "user", "content": prompt}]
        
        result = codex_project.validate(
            messages=messages,
            query=query,
            context=context,
            response=response,
        )
        
        trust_score = result.model_dump()["eval_scores"]["trustworthiness"]["score"]
        
        if result.expert_answer and result.escalated_to_sme:
            final_response = result.expert_answer
        elif result.should_guardrail:
            final_response = "Response flagged as potentially unreliable."
        else:
            final_response = response
        
        return {
            "trust_score": float(trust_score),
            "validated_response": final_response,
            "guardrailed": result.should_guardrail,
        }
    except Exception as e:
        print(f"Codex validation skipped: {e}")
        return {"trust_score": None, "validated_response": response, "guardrailed": False}


# Test trust scoring (requires valid CODEX_API_KEY)
rag_result = query_documents("What are the recommendations for climate change?")
trust_result = validate_with_codex(
    query="What are the recommendations for climate change?",
    context=rag_result["context"],
    response=rag_result["response"]
)

print(f"Response: {trust_result['validated_response']}")
if trust_result['trust_score']:
    score_pct = trust_result['trust_score'] * 100
    emoji = '🟢' if score_pct >= 70 else '🟡' if score_pct >= 50 else '🔴'
    print(f"Trust Score: {emoji} {score_pct:.1f}%")
else:
    print("Trust Score: Not available (Codex API key needed)")

## 7. Complete Router Pipeline

Bringing it all together — the full pipeline that classifies, routes, and responds to any query.

In [ ]:
def process_query(query: str) -> dict:
    """Complete query processing pipeline with intelligent routing."""
    # Step 1: Classify
    route = classify_query(query)
    
    # Step 2: Route and process
    if route == "sql":
        result = text_to_sql(query)
        return {
            "response": result["response"],
            "route": "sql",
            "sql_query": result["sql_query"],
            "trust_score": None,
        }
    else:
        rag_result = query_documents(query)
        trust_result = validate_with_codex(
            query=query,
            context=rag_result["context"],
            response=rag_result["response"]
        )
        return {
            "response": trust_result["validated_response"],
            "route": "rag",
            "sources": rag_result["sources"],
            "trust_score": trust_result["trust_score"],
        }


print("Router pipeline ready!")

In [ ]:
# Test the complete pipeline
test_cases = [
    "What is the population of Houston, Texas?",
    "What are the key findings about climate change?",
    "Which state has the most cities in the database?",
    "What does the report recommend for emissions?",
]

print("=" * 70)
print("INTELLIGENT QUERY ROUTER - END-TO-END TEST")
print("=" * 70)

for query in test_cases:
    print(f"\n{'─' * 70}")
    print(f"Query: {query}")
    result = process_query(query)
    
    route_icon = "🗄️ SQL" if result["route"] == "sql" else "📄 RAG"
    print(f"Route: {route_icon}")
    print(f"Response: {result['response']}")
    
    if result.get("sql_query"):
        print(f"SQL: {result['sql_query']}")
    if result.get("trust_score"):
        print(f"Trust: {result['trust_score']*100:.1f}%")
    if result.get("sources"):
        print(f"Sources: {result['sources']}")

## 8. Architecture Summary

```
User Query
    │
    ▼
┌──────────────────────┐
│  GPT-4o-mini Router  │ ─── Classifies intent (sql/rag)
└──────────┬───────────┘
           │
     ┌─────┴─────┐
     │           │
     ▼           ▼
┌─────────┐  ┌──────────────┐
│SQL Path │  │  RAG Path    │
│         │  │              │
│ NL→SQL  │  │ ChromaDB     │
│ Execute │  │ Retrieval    │
│ Synth   │  │ Generation   │
└────┬────┘  └──────┬───────┘
     │              │
     │              ▼
     │       ┌──────────────┐
     │       │Codex Scoring │
     │       └──────┬───────┘
     │              │
     └──────┬───────┘
            ▼
     ┌──────────────┐
     │   Response   │
     └──────────────┘
```